# Predviđanje zarade od prodaje mješovite robe
### Projekat iz predmeta Vještačka inteligencija i mašinsko učenje

**Student:** Jovan Popović  
**Broj indeksa:** 25/014  
**Email:** jovan.popovic3@udg.edu.me  
**Godina:** Master 1, FIST, UDG

---

Cilj projekta je da, na osnovu istorijskih maloprodajnih transakcija, predvidi ostvarenu zaradu po transakciji `sales_amount = quantity · unit_price · (1 − discount_pct/100)`.
Iz feature seta isključujemo `unit_price` kako problem ne bi bio trivijalno rješiv samom formulom, čime model mora da nauči tipičnu cijenu iz kategorije proizvoda, brenda, regiona i ostalih atributa.

Notebook prati sljedeće faze:

1. Učitavanje i pregled skupa podataka  
2. Otkrivanje i vizuelizacija podataka  
3. Priprema podataka za algoritme mašinskog učenja  
4. Obuka i procjena regresionih modela  
5. Generisanje višeklasne etikete iz numeričke ciljne varijable  
6. Obuka i procjena klasifikacionih modela  
7. Zaključak


## 0. Pripremni korak – Colab okruženje

Sljedeća ćelija prepoznaje da li se notebook izvršava lokalno ili u Google Colab-u i u oba slučaja postavlja ispravne putanje do `src/` modula i dataset-a. Ako se izvršava u Colab-u, učitava ovaj repozitorijum sa GitHub-a (alternativno, otpremite `projekat-retail-sales.zip` ručno).

In [ ]:
# Detekcija okruženja i postavljanje putanja
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Kada se izvršava u Colab-u: učitati cijeli folder iz Drive-a ili rasporediti
    # zip arhivu manuelno. Ovdje pretpostavljamo da je projekat otpakovan u
    # /content/projekat-retail-sales/.
    PROJECT_ROOT = "/content/projekat-retail-sales"
    if not os.path.exists(PROJECT_ROOT):
        print("⚠ Folder projekta nije pronađen. Otpremi 'projekat-retail-sales.zip'")
        print("  i raspakuj ga u /content/, ili poveži Google Drive ako se tamo nalazi.")
else:
    # Lokalno: notebook se izvršava iz korijena projekta
    PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()

sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
DATA_PATH = os.path.join(PROJECT_ROOT, "datasets", "retail_sales_dataset.csv")
OUT_PATH  = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUT_PATH, exist_ok=True)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_PATH    = {DATA_PATH}")


In [ ]:
# Standardni importi
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Naši moduli
from data_loader import (load_dataset, drop_unused_columns,
                         add_synthetic_missing, stratified_split,
                         split_features_target, build_preprocessor,
                         make_classification_target,
                         TARGET_COL, NUMERICAL_COLS, CATEGORICAL_COLS)
from visualization import (plot_histograms, plot_correlation_matrix,
                            plot_combined_attributes,
                            plot_category_target_boxplot,
                            plot_learning_curve,
                            plot_regression_metrics_bar,
                            plot_confusion_matrices,
                            plot_classification_metrics_bar,
                            plot_pr_curves_multiclass,
                            plot_roc_curves_multiclass)
from regression_models import (build_regressors,
                               train_and_evaluate_all as train_regressors,
                               metrics_to_dataframe as reg_metrics_df)
from classification_models import (build_classifiers,
                                    train_and_evaluate_all as train_classifiers,
                                    metrics_to_dataframe as cls_metrics_df)

# Globalna podešavanja
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
RANDOM_STATE = 14
SAMPLE_SIZE  = 10_000


## 1. Učitavanje skupa podataka

Originalni dataset `retail_sales_dataset.csv` sadrži 120 000 transakcija. Za potrebe efikasne obuke svih modela (uključujući SVR sa polinomijalnim i RBF jezgrom), radimo stratifikovano sub-sampling na 10 000 redova — slojevi su kvartili `sales_amount` ciljne varijable, čime se očuvava originalna distribucija.

In [ ]:
df_full = pd.read_csv(DATA_PATH)
print(f"Originalni skup: {df_full.shape[0]} redova × {df_full.shape[1]} kolona")

df = load_dataset(DATA_PATH, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE)
print(f"Radni (sub-sample) skup: {df.shape[0]} redova × {df.shape[1]} kolona")


## 2. Otkrivanje i vizuelizacija podataka

### 2.1 Tabelarni prikaz izvornog skupa

In [ ]:
df.head(10)

### 2.2 Osnovne statističke karakteristike

In [ ]:
print("Tipovi podataka i broj nedostajućih vrijednosti:")
info_table = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nedostaje": df.isna().sum(),
    "jedinstvenih": df.nunique(),
})
info_table

In [ ]:
print("Deskriptivne statistike numeričkih kolona:")
df.describe().round(2)

In [ ]:
print("Deskriptivne statistike kategoričkih kolona:")
df.describe(include="object").T

### 2.3 Histogrami numeričkih atributa

In [ ]:
numeric_for_hist = ["quantity", "discount_pct", "unit_price", "sales_amount"]
plot_histograms(df, numeric_for_hist,
                save_path=os.path.join(OUT_PATH, "histograms.png"))

Histogrami pokazuju karakteristike kontinualnih atributa: `quantity` ima diskretan opseg 1–5 sa dominacijom 1, `discount_pct` je diskretna varijabla sa nekoliko nivoa (0% je najčešći), `unit_price` je relativno uniformna, dok `sales_amount` ima pozitivno iskrivljenu distribuciju jer veliki proizvodi i veće količine generišu rep.

### 2.4 Matrica korelacije

In [ ]:
plot_correlation_matrix(df, numeric_for_hist,
                        save_path=os.path.join(OUT_PATH, "correlation.png"))

Najjača korelacija je očekivana — `unit_price` × `quantity` × (1−`discount_pct`/100) deterministički daje `sales_amount`, pa je korelacija `unit_price`↔`sales_amount` značajna. Negativna korelacija `discount_pct`↔`sales_amount` je takođe očekivana.

### 2.5 Smislena kombinacija atributa

In [ ]:
# (a) Scatter quantity × unit_price obojen po kategoriji
plot_combined_attributes(df, x="quantity", y="unit_price", hue="category",
                         save_path=os.path.join(OUT_PATH, "combo_qty_price_by_cat.png"))

In [ ]:
# (b) Box plot: sales_amount po kategoriji proizvoda
plot_category_target_boxplot(df, category="category", target="sales_amount",
                             save_path=os.path.join(OUT_PATH, "box_sales_by_category.png"))

In [ ]:
# (c) Box plot: sales_amount po segmentu kupca
plot_category_target_boxplot(df, category="customer_segment", target="sales_amount",
                             save_path=os.path.join(OUT_PATH, "box_sales_by_segment.png"))

Iz ovih grafikona se može uočiti da:
- različite kategorije proizvoda (Electronics, Sports, Clothing…) imaju izrazito različite cjenovne raspone, što direktno utiče i na `sales_amount`;
- segmenti kupaca (`New`, `Returning`, `Loyal`, `VIP`) ne pokazuju dramatičnu razliku u zaradi po transakciji — varijacija je manja od one koja proizlazi iz kategorije proizvoda.

## 3. Priprema podataka za mašinsko učenje

Pripremni koraci:
1. Izbacujemo identifikacijske kolone i `unit_price` (sprečavanje target leakage-a).
2. Sintetički ubacujemo ~3% NaN vrijednosti u dvije kolone da bismo demonstrirali rad sa nedostajućim vrijednostima.
3. Stratifikovana 80/20 podjela na trening i test (slojevi po kvartilima `sales_amount`-a).
4. Razdvajanje X (prediktori) i y (target).
5. ColumnTransformer pipeline koji imputira nedostajeće vrijednosti, skalira numeričke i one-hot enkodira kategoričke atribute.

In [ ]:
# 3.1 Pripremni filteri
df_prep = drop_unused_columns(df)
print(f"Nakon izbacivanja ID i unit_price kolone: {df_prep.shape}")

# 3.2 Demonstracija nedostajućih vrijednosti
df_prep = add_synthetic_missing(df_prep,
                                 columns=["quantity", "customer_segment"],
                                 fraction=0.03,
                                 random_state=RANDOM_STATE)
print(f"Nedostajuće vrijednosti nakon ubacivanja:\n{df_prep.isna().sum()[df_prep.isna().sum() > 0]}")


In [ ]:
# 3.3 Stratifikovana podjela
train_df, test_df = stratified_split(df_prep, test_size=0.2,
                                      random_state=RANDOM_STATE)
print(f"Trening: {train_df.shape}, Test: {test_df.shape}")

# 3.4 Razdvajanje prediktora i targeta
X_train, y_train = split_features_target(train_df)
X_test,  y_test  = split_features_target(test_df)
print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")


In [ ]:
# 3.5 Preprocessor + fit_transform
preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"Nakon predobrade, broj feature-a: {X_train_processed.shape[1]}")
print(f"  - numeričkih (scaler): {len(NUMERICAL_COLS)}")
print(f"  - kategoričkih (one-hot): {X_train_processed.shape[1] - len(NUMERICAL_COLS)}")


## 4. Regresioni modeli – predviđanje `sales_amount`

Obuka i evaluacija sljedećih regresora:
- Linearni i polinomijalni (stepen 2) regresor  
- SGD regresor  
- Regularizovani: Ridge, Lasso, Elastična mreža  
- SVR sa linearnim, polinomijalnim (st. 2) i RBF jezgrom  
- Stablo odlučivanja  
- Random Forest – bagging (bootstrap=True) i pasting (bootstrap=False)

Metrika: **RMSE** i **MAE** na test skupu.

In [ ]:
regressors = build_regressors(random_state=RANDOM_STATE)
print(f"Ukupno regresionih modela za obuku: {len(regressors)}\n")

reg_metrics, reg_fitted = train_regressors(
    regressors, X_train_processed, y_train, X_test_processed, y_test)


In [ ]:
reg_results = reg_metrics_df(reg_metrics)
reg_results

In [ ]:
plot_regression_metrics_bar(reg_metrics,
                            save_path=os.path.join(OUT_PATH, "regression_metrics.png"))

### 4.1 Krive učenja

Krive učenja prikazuju kako greška na trening i validacijskom skupu evoluira sa povećanjem broja trening instanci. Veliki razmak između krivih ukazuje na overfitting, a obje krive sa visokom greškom — underfitting.

In [ ]:
# Krive učenja za odabrane modele - one koji su brzi za learning_curve
selected_for_lc = [
    "Linearna regresija",
    "Ridge regresija",
    "SGD regresor",
    "Stablo odlučivanja",
    "Random Forest (bagging)",
]
for name in selected_for_lc:
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    plot_learning_curve(
        regressors[name].__class__(**regressors[name].get_params()) if not hasattr(regressors[name], "steps")
            else regressors[name],
        X_train_processed, y_train, title=name,
        save_path=os.path.join(OUT_PATH, f"lc_reg_{safe_name}.png"))


## 5. Generisanje višeklasne kategoričke etikete

Iz numeričkog `sales_amount`-a kreiramo četiri klase prema kvartilima distribucije:
- 0 = **Niska** zarada (najniži kvartil)  
- 1 = **Niža-srednja** zarada  
- 2 = **Viša-srednja** zarada  
- 3 = **Visoka** zarada (najviši kvartil)

In [ ]:
# Granice se računaju na trening skupu, primjenjuju i na test
y_train_class, class_names, bin_edges = make_classification_target(y_train, n_classes=4)

# Proširi granice na -inf/+inf da test vrijednosti van trening opsega ne daju NaN
import numpy as np
edges_ext = np.concatenate(([-np.inf], bin_edges[1:-1], [np.inf]))
y_test_class = pd.cut(y_test, bins=edges_ext, labels=range(4),
                       include_lowest=True).astype(int)

print("Granice kvartila (€):", [round(b, 2) for b in bin_edges])
print("\nRaspodjela klasa u trening skupu:")
print(y_train_class.value_counts().sort_index().to_frame(name="count"))
print("\nRaspodjela klasa u test skupu:")
print(y_test_class.value_counts().sort_index().to_frame(name="count"))
print("\nNazivi klasa:", class_names)


## 6. Klasifikacioni modeli – predviđanje kategorije zarade

Obuka i evaluacija sljedećih klasifikatora:
- SGD klasifikator
- Logistički regresor
- SVC sa linearnim, polinomijalnim (st. 2) i RBF jezgrom
- Stablo odlučivanja
- Random Forest – bagging i pasting

Metrike: **accuracy**, **precision (macro)**, **recall (macro)**, **F1 (macro)** + matrica konfuzije + PR/ROC kriva.

In [ ]:
classifiers = build_classifiers(random_state=RANDOM_STATE)
print(f"Ukupno klasifikacionih modela za obuku: {len(classifiers)}\n")

cls_metrics, cls_predictions, cls_probabilities, cls_fitted = train_classifiers(
    classifiers, X_train_processed, y_train_class,
    X_test_processed, y_test_class)


In [ ]:
cls_results = cls_metrics_df(cls_metrics)
cls_results

In [ ]:
plot_classification_metrics_bar(cls_metrics,
                                save_path=os.path.join(OUT_PATH, "classification_metrics.png"))

### 6.1 Matrice konfuzije

In [ ]:
plot_confusion_matrices(y_test_class, cls_predictions, class_names,
                        save_path=os.path.join(OUT_PATH, "confusion_matrices.png"))

### 6.2 PR i ROC krive (mikro-prosjek)

In [ ]:
plot_pr_curves_multiclass(y_test_class, cls_probabilities, n_classes=4,
                          save_path=os.path.join(OUT_PATH, "pr_curves.png"))

In [ ]:
plot_roc_curves_multiclass(y_test_class, cls_probabilities, n_classes=4,
                           save_path=os.path.join(OUT_PATH, "roc_curves.png"))

## 7. Zaključak

Projekat je realizovao kompletan pipeline mašinskog učenja na realističnom maloprodajnom skupu podataka. Posmatrajući stvarne rezultate na test skupu od 2 000 transakcija (sub-sample od 10 000), zaključci su sljedeći:

**Regresioni dio:** Linearni i regularizovani regresori (Linear, Ridge, Lasso, Elastic Net, SGD) postigli su gotovo identičan RMSE od oko 275 € i MAE od oko 200 €, što su najbolji rezultati u ovom eksperimentu. Razlog je što je, nakon uklanjanja `unit_price`-a, najveći dio signala u podacima dolazi iz aditivne kombinacije kategoričkih atributa (kategorija proizvoda, brend, region, popust, količina), koju linearni model nakon one-hot kodiranja efikasno hvata. Random Forest se približava (RMSE ≈ 282), dok SVR sa polinomijalnim i RBF jezgrom bez hyperparameter tuning-a daje slabije rezultate (RMSE 309–341) — RBF kernel je posebno osjetljiv na izbor hiperparametara `C` i `gamma` i bez grid search-a teško nadmašuje linearne metode.

**Klasifikacioni dio:** Najbolji F1 (makro-prosjek) od ~0,50 postigli su Random Forest (bagging) i SVC sa RBF jezgrom, dok su linearni klasifikatori (SGD, Logistička, linearni SVC) ostali na F1 ≈ 0,43–0,44. Klasifikaciona tačnost od ~51% na 4 uravnotežene klase znatno je iznad random baseline-a (25%), ali takođe pokazuje koliko je signal u feature-ima ograničen kada se isključi `unit_price`. PR i ROC krive pokazuju da svi nelinearni modeli (Stablo, RF, SVC poly/RBF) imaju veću AUC vrijednost od linearnih.

**Generalizacija – krive učenja:** Linearni modeli vrlo brzo dostižu plateau na ~275 RMSE već sa 2 000 trening instanci, što ukazuje da im više podataka ne pomaže — pomogla bi bogatija feature representation (interakcijski atributi, target encoding). Random Forest pokazuje sporo opadanje validacijske greške, dok stablo odlučivanja sa dubinom 8 pokazuje znakove overfitting-a (trening krivulja je značajno ispod validacijske).

**Šta dalje:** Mogući pravci poboljšanja:
1. Feature engineering — kreiranje interakcijskih atributa (`category × region`, `brand × segment`), te ekstrakcija sezonskih signala iz `transaction_date` (mjesec, dan u sedmici);
2. Hyperparameter tuning preko `GridSearchCV` ili `RandomizedSearchCV`, naročito za SVR RBF (`C`, `gamma`) i Random Forest (`n_estimators`, `max_depth`, `min_samples_split`);
3. Gradient boosting modeli (`XGBoost`, `LightGBM`, `CatBoost`) koji često nadmašuju i Random Forest na tabularnim podacima sa mješavinom numeričkih i kategoričkih feature-a;
4. Eksperimentisanje sa target enkodiranjem kategoričkih atributa (umjesto one-hot), naročito za `category` i `region` koji imaju snažan signal o tipičnoj cijeni proizvoda.
